# Bit Manipulation

*All Operators · XOR Tricks · Subset Masks · Real-World*


---
## Bit Manipulation


# Bit Manipulation

*Run each cell with **Shift+Enter***

01 — DSA Internals: Bit Manipulation Toolkit
============================================

Runnable companion to PDF Book II "Thinking in bits".

Integers are binary under the hood. Bit tricks turn certain O(n) or extra-space
problems into O(1) space / branch-free operations, and they show up constantly
in interviews (subsets, flags, low-level protocols, hashing).

Core operators:
    &  AND    |  OR    ^  XOR    ~  NOT    <<  left shift    >>  right shift

Key identities used below:
    x ^ x == 0,   x ^ 0 == x          -> XOR cancels pairs (find the loner)
    x & (x - 1)                       -> clears the lowest set bit
    x & -x                            -> isolates the lowest set bit
    x & (x - 1) == 0                  -> x is a power of two (for x > 0)


---
## 🧠 Mental Model: Bit Manipulation

> **Integers are binary arrays. Bit operations let you manipulate individual bits in O(1) with no extra memory.**  
> They turn O(n) space problems into O(1), and O(n) time problems into constant-time with masking tricks.

### WHY — Why does it exist?
CPUs operate natively on bits. Bit tricks give:
- **O(1) space**: represent a set of n booleans as one integer
- **Branch-free code**: swap without a temp, conditionals as masks
- **Interview signal**: shows you understand what's under the hood

### WHAT — Operators & key identities

| Operator | Symbol | Example | What it does |
|----------|--------|---------|-------------|
| AND | `&` | `5 & 3 = 1` | Keep bits set in BOTH |
| OR | `\|` | `5 \| 3 = 7` | Keep bits set in EITHER |
| XOR | `^` | `5 ^ 3 = 6` | Keep bits set in EXACTLY ONE |
| NOT | `~` | `~5 = -6` | Flip all bits (two's complement) |
| Left shift | `<<` | `1 << 3 = 8` | Multiply by 2ⁿ |
| Right shift | `>>` | `8 >> 2 = 2` | Divide by 2ⁿ (floor) |

**Key identities:**
```
x ^ x == 0          → XOR cancels pairs  (find single loner)
x ^ 0 == x          → XOR with 0 is identity
x & (x-1) == 0      → x is a power of 2 (for x > 0)
x & (x-1)           → clears the lowest set bit of x
x & (-x)            → isolates the lowest set bit of x
x >> k & 1          → gets the k-th bit of x
x | (1 << k)        → sets the k-th bit of x
x & ~(1 << k)       → clears the k-th bit of x
x ^ (1 << k)        → toggles the k-th bit of x
```

### HOW — Common patterns

**Find the single element (all others appear twice):**
```python
result = 0
for x in nums: result ^= x   # pairs cancel; lone survivor remains
```

**Count set bits (Brian Kernighan):**
```python
count = 0
while x: x &= x - 1; count += 1   # each step removes lowest set bit
```

**Generate all subsets of n elements:**
```python
for mask in range(1 << n):          # 2ⁿ masks
    subset = [i for i in range(n) if mask >> i & 1]
```

**Check if kth bit is set:**
```python
is_set = bool(x >> k & 1)
```

### WHEN — Use cases

| Problem | Bit trick |
|---------|-----------|
| Detect single non-duplicate | XOR all elements |
| Check power of two | `x > 0 and x & (x-1) == 0` |
| Count set bits | Brian Kernighan / `bin(x).count("1")` |
| Represent a subset | Bitmask over n elements (n ≤ 30) |
| Permission flags | OR to grant, AND NOT to revoke, AND to check |
| Swap without temp | `a ^= b; b ^= a; a ^= b` (but avoid; unclear) |

**Gotchas:**
- Python integers are arbitrary precision — `~x == -(x+1)` due to two's complement.
- `1 << n` can overflow in C/Java (use `1 << 63` carefully); NOT an issue in Python.
- XOR swap is clever but error-prone in interviews — prefer a temporary variable.
- `x & -x` works because Python uses two's complement for negative numbers.


In [ ]:
from __future__ import annotations


def get_bit(x: int, i: int) -> int:
    """Return bit i (0 = least significant)."""
    return (x >> i) & 1


def set_bit(x: int, i: int) -> int:
    return x | (1 << i)


def clear_bit(x: int, i: int) -> int:
    return x & ~(1 << i)


def toggle_bit(x: int, i: int) -> int:
    return x ^ (1 << i)


def count_set_bits(x: int) -> int:
    """Brian Kernighan's algorithm: loops once per set bit, not per bit."""
    count = 0
    while x:
        x &= x - 1                          # drop the lowest set bit
        count += 1
    return count


def is_power_of_two(x: int) -> bool:
    return x > 0 and (x & (x - 1)) == 0


def lowest_set_bit(x: int) -> int:
    """Isolate the lowest set bit, e.g. 0b10110 -> 0b00010."""
    return x & -x


def swap_without_temp(a: int, b: int) -> tuple[int, int]:
    a ^= b
    b ^= a
    a ^= b
    return a, b


def single_number(nums: list[int]) -> int:
    """Every element appears twice except one — XOR cancels the pairs. O(1) space."""
    result = 0
    for n in nums:
        result ^= n
    return result


def all_subsets(items: list) -> list[list]:
    """Enumerate the 2**n subsets: bit j of the counter = include items[j]."""
    n = len(items)
    out: list[list] = []
    for mask in range(1 << n):
        subset = [items[j] for j in range(n) if mask & (1 << j)]
        out.append(subset)
    return out


def demo() -> None:
    # Single-bit operations.
    assert get_bit(0b1010, 1) == 1 and get_bit(0b1010, 0) == 0
    assert set_bit(0b1000, 0) == 0b1001
    assert clear_bit(0b1011, 1) == 0b1001
    assert toggle_bit(0b1010, 2) == 0b1110
    print("   get/set/clear/toggle single bits verified")

    # Population count and power-of-two.
    assert count_set_bits(0b10110110) == 5
    assert count_set_bits(255) == 8
    assert [is_power_of_two(n) for n in (1, 2, 3, 4, 16, 17)] == [True, True, False, True, True, False]
    print("   count_set_bits(0b10110110) =", count_set_bits(0b10110110), " is_power_of_two ✔")

    # Isolate lowest set bit & XOR swap.
    assert lowest_set_bit(0b10110) == 0b00010
    assert swap_without_temp(7, 42) == (42, 7)
    print("   lowest_set_bit + XOR swap verified")

    # Classic interview: the number that appears once.
    assert single_number([4, 1, 2, 1, 2]) == 4
    assert single_number([7]) == 7
    print("   single_number([4,1,2,1,2]) =", single_number([4, 1, 2, 1, 2]), " (O(1) space via XOR)")

    # Subset enumeration via bitmask.
    subs = all_subsets(["a", "b", "c"])
    assert len(subs) == 8 and [] in subs and ["a", "b", "c"] in subs
    print("   all_subsets(['a','b','c']) ->", len(subs), "subsets (2**3)")


def main() -> None:
    print("=" * 70)
    print("DSA INTERNALS — bit_manipulation.py")
    print("=" * 70)
    print("Branch-free / O(1)-space tricks with & | ^ ~ << >>:")
    demo()
    print("-" * 70)
    print("Lesson: XOR cancels pairs; x&(x-1) clears the lowest bit; a bitmask enumerates all subsets.")
    print("All bit_manipulation demos passed ✔")


if __name__ == "__main__":
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()